# Phase 4: Statistical Analysis

Validating EDA findings using descriptive statistics, parametric/non-parametric tests, and hypothesis testing to confirm key churn drivers.

**Tests Executed:**
1. **Chi-Square Test of Independence**: Categorical variables (`contract`, `payment_method`, `tech_support`) vs. `churn`.
2. **Two-Sample T-Test / Mann-Whitney U**: Continuous variables (`monthly_charges`, `tenure`) across churn groups.
3. **95% Confidence Intervals**: Difference in average spend between churned and retained customers.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load clean dataset
df = pd.read_csv("../data/processed/customer_churn_clean.csv")

target_col = 'churn_label' if 'churn_label' in df.columns else 'churn'
print(f"Loaded {len(df)} rows.")

Loaded 7043 rows.


### 1. Summary Statistics by Churn Status
Comparing numerical features (Mean, Median, Standard Deviation, Skewness) across churned vs. retained customers.

In [2]:
num_cols = ['monthly_charges', 'total_charges']
if 'tenure_months' in df.columns:
    num_cols.append('tenure_months')
elif 'tenure' in df.columns:
    num_cols.append('tenure')

# Grouped summary stats
summary_stats = df.groupby(target_col)[num_cols].agg(['mean', 'median', 'std', 'skew']).round(2)
summary_stats

monthly_charges                     total_charges           \
                       mean median    std  skew          mean   median   
churn_label                                                              
No                    61.27  64.43  31.09 -0.03       2549.91  1679.52   
Yes                   74.44  79.65  24.67 -0.73       1531.80   703.55   

                           tenure_months                      
                 std  skew          mean median    std  skew  
churn_label                                                   
No           2329.95  0.81         37.57   38.0  24.11 -0.03  
Yes          1890.82  1.51         17.98   10.0  19.53  1.15

### 2. Hypothesis Testing: Categorical Features (Chi-Square Test)
* **$H_0$ (Null Hypothesis)**: Feature and Churn are independent (no association).
* **$H_1$ (Alternative Hypothesis)**: Feature and Churn are significantly associated.

In [3]:
cat_cols = ['contract', 'payment_method', 'internet_service', 'tech_support', 'paperless_billing']

chi2_results = []

for col in cat_cols:
    if col in df.columns:
        contingency_table = pd.crosstab(df[col], df[target_col])
        chi2, p, dof, _ = stats.chi2_contingency(contingency_table)
        chi2_results.append({
            'Feature': col,
            'Chi2 Statistic': round(chi2, 2),
            'p-value': p,
            'Significant (α=0.05)': p < 0.05
        })

chi2_df = pd.DataFrame(chi2_results)
chi2_df

,Feature,Chi2 Statistic,p-value,Significant (α=0.05)
0,contract,1184.60,5.863038e-258,True
1,payment_method,648.14,3.682355e-140,True
2,internet_service,732.31,9.571788e-160,True
3,tech_support,190.17,2.923567e-43,True
4,paperless_billing,258.28,4.073355e-58,True


### 3. Hypothesis Testing: Numerical Features (T-Test & Mann-Whitney U)
Testing whether monthly charges and tenure differ significantly between churned and active customers.

In [4]:
churned = df[df[target_col].isin([1, 'Yes', 'True'])]
retained = df[df[target_col].isin([0, 'No', 'False'])]

# T-Test for Monthly Charges
t_stat, p_val_m = stats.ttest_ind(churned['monthly_charges'], retained['monthly_charges'])

# Mann-Whitney U for Tenure (often non-normal)
tenure_var = 'tenure_months' if 'tenure_months' in df.columns else 'tenure'
u_stat, p_val_t = stats.mannwhitneyu(churned[tenure_var], retained[tenure_var])

print(f"Monthly Charges T-Test: t = {t_stat:.2f}, p-value = {p_val_m:.4e}")
print(f"Tenure Mann-Whitney U:  U = {u_stat:.2f}, p-value = {p_val_t:.4e}")

Monthly Charges T-Test: t = 16.54, p-value = 2.7066e-60
Tenure Mann-Whitney U:  U = 2515538.00, p-value = 2.4196e-208


### 4. 95% Confidence Interval for Mean Difference in Monthly Charges

In [5]:
# Calculate difference in mean monthly charges
mean_diff = churned['monthly_charges'].mean() - retained['monthly_charges'].mean()
se_diff = np.sqrt((churned['monthly_charges'].var() / len(churned)) + (retained['monthly_charges'].var() / len(retained)))

ci_lower = mean_diff - (1.96 * se_diff)
ci_upper = mean_diff + (1.96 * se_diff)

print(f"Mean Difference: ${mean_diff:.2f}")
print(f"95% Confidence Interval: [${ci_lower:.2f}, ${ci_upper:.2f}]")

Mean Difference: $13.18
95% Confidence Interval: [$11.77, $14.58]
